# NCAA Rowing Roster Extraction

This project is a summary of my time as a Division I rower at Duquesne University. Over four years of competing in the Atlantic 10 conference I had many meets and many more teammates. The work here uses various analysis tools (**webscraping, string manipulation, visualizations**) to demonstrate and tell the narrative of my collegiate carrer.

In [22]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import csv

## Stage 1: Collecting general rower information

Names, photos, and roster webpages

Method to request access to the Duquesne athletics site

In [2]:
def request_access(finish_url:str):
    """Make site request for webscraping data

    Args:
        finish_url: ending of web address depending on info desired

    Returns:
        r: request message (200 indicating success)
    """
    headers = {'user-agent': 'H Valenty personal study (unc6kr@virginia.edu)'}
    r = requests.get(f"https://goduquesne.com/sports/womens-rowing/roster/{finish_url}",
                     headers = headers)
    return r

Method to clean front and back end of entries

In [3]:
def remove_str_start_end(s, start, end):
    """string function to clean formatted entries"""
    return s[:start] + s[end + 1:]

Method for primary cleaning of webscraped data

In [4]:
def initial_clean(req):
    """cleaning gathered data to remove
    javascript formatting and code

    Args:
        req: request response of website data

    Returns:
        rep: cleaned data as list
    """
    # access javascript code from request to website
    roster = BeautifulSoup(req.text, 'html').find_all('script', type=True)[0]
    # format javascript as string
    str_roster = str(roster)
    # split string to isolate each rower
    roster_list = str_roster.split("Person")[1:]
    # clean formatted text from each rower
    clean = [a.strip('",\"@context":\"http://schema.org\",') for a in roster_list ]
    # call method to further clean start and end of entries
    rep = [i.strip(remove_str_start_end(clean[0], 31, -9)) for i in clean]

    return rep

Method to transform data into formatted dataframe

In [5]:
def dataframe_format(rep):
    """alter list data to more workable format as dataframe
    Args:
        rep: cleaned data in list format

    Returns:
        duq_roster: pandas dataframe of cleaned data
    """
    # fully split text into list of lists for rower entries
    hold = [rep[i].split(',') for i in range(len(rep))]
    # format into pandas dataframre and selection columns of interest
    duq_roster = pd.DataFrame(hold)[[0,3,4,5]]
    # rename columns 
    replace_map = {0:'photo_url',
              3:'athlete',
              4:'gender',
              5:'page_url'}

    duq_roster = duq_roster.rename(replace_map, axis = 1)
    # drop nulls from dataframe
    duq_roster = duq_roster.drop(duq_roster[duq_roster.photo_url == 'null'].index).reset_index(drop=True)

    return duq_roster

End process cleaning for formatted dataframe

In [6]:
def final_clean(duq_roster):
    """combination of string regex for final cleaning"""
    duq_roster['photo_url'] = [duq_roster.photo_url[i].replace('url":"', '') for i in range(len(duq_roster))]
    duq_roster.athlete = [duq_roster.athlete[i].replace('"name":"', '').strip('"') for i in range(len(duq_roster))]
    duq_roster.gender = [duq_roster.gender[i].replace('"gender":"', '').strip('"') for i in range(len(duq_roster))]
    duq_roster.page_url = [duq_roster.page_url[i].replace('"url":"', '') for i in range(len(duq_roster))]

    return duq_roster

Execute methods for 4 seasons of collegiate career

In [7]:
seasons = ['2020-21','2021-22','2022-23','2023-24']
season_df = {}
# iterate through all 4 seasons to generate data tables
for season in seasons:
    r = request_access(season)
    rep = initial_clean(req=r)
    duq_roster = dataframe_format(rep=rep)
    season_df["season-{0}".format(season)] = final_clean(duq_roster=duq_roster)

Show snippet of 2020-21 season roster

In [8]:
season_df['season-2020-21'].tail(6)

,photo_url,athlete,gender,page_url
51,https://goduquesne.com/images/2020/11/17/Hanna...,Hannah Valenty,F,https://goduquesne.com/roster.aspx?rp_id=10481
52,https://goduquesne.com/images/2019/8/1/New_D_h...,Anna Vignali,F,https://goduquesne.com/roster.aspx?rp_id=10495
53,https://goduquesne.com/images/2020/11/17/Zara_...,Zara Wenzinger,F,https://goduquesne.com/roster.aspx?rp_id=10472
54,https://goduquesne.com/images/2020/11/17/Britt...,Britta Wheeler,F,https://goduquesne.com/roster.aspx?rp_id=10496
55,https://goduquesne.com/images/2019/8/1/New_D_h...,Madelyn Winiarski,F,https://goduquesne.com/roster.aspx?rp_id=10497
56,https://goduquesne.com/images/2020/11/17/Grace...,Grace Yeretzian,F,https://goduquesne.com/roster.aspx?rp_id=10479...


## Stage 2: Rower web-id information

Extracting rower web-id, lastname, and firstname

Find the web address values for 2023-24 roster members (name + web id)

In [9]:
def rower_id_compile(req):    
# text from web request for web ids and rower names
    option = BeautifulSoup(req.text, 'html').find("option", {"value": True})
    option_str = str(option).split('<option ')
    # regex formatting after converting to string
    option_df = pd.DataFrame(map(lambda o: o.replace('selected="selected" ', '').replace('value=', '')\
                                .replace('"', '').split('\r\n')[0], option_str))[1:]
    # split into separate dataframe columns
    option_df[['web_id','rower']] = option_df[0].str.split('>', n=1, expand=True)
    option_df[['last_name','first_name']] = option_df['rower'].str.split(',', n=1, expand=True)
    # select final dataframe columns, removing originals
    option_df = option_df[['web_id','last_name','first_name']]
    # convert web id to numeric type
    option_df['web_id'] = pd.to_numeric(option_df['web_id'])

    return option_df

Choose 4 rowers who graduated each year from 2021-2024, collect id and rower names

In [10]:
grad_rowers = ['aleiia-asmundson/10436', 'kayla-eads/11122', 'madison-barker/11534', 'hannah-valenty/12209']
rower_id_df = {}
id_season = 2021
# iterate through all 4 seasons to generate data tables
for rower in grad_rowers:
    r = request_access(rower)
    rower_id_df["season-{0}".format(id_season)] = rower_id_compile(req=r)
    id_season+=1

Snippet of one season

In [11]:
rower_id_df['season-2024'].tail()

,web_id,last_name,first_name
47,12253,Tziovannis,Kyra
48,12209,Valenty,Hannah
49,12154,van der Net,Rosemary
50,12192,Wheeler,Britta
51,12255,Woroszylo,Nicole


Now have 2 dictionaries of dataframes

* `season_df`: (4 seasons) photo_url, athlete, gender, page_url
* `rower_id_df`: (4 seasons) web_id, last_name, first_name
    * Need to join tables to condense overlap while maintaining graduation year

Use name + web id to get access to personal pages (easier to condense and automate than page_url). Will give final rower bio information.

* `bio_df`: athlete, high school, hometown

## Stage 3: Rower biographical information

Extracting rower names, class (to be transformed to graduation year), highschool, and hometown

Thought process

* get rower web-id and their name to input into web address format
* have to access each rower page for their bio info (class, high school, hometown)
* the personal pages capture for a moment in time the last year the rower was on the team
    * need to access 4 pages where a rower was a senior to get all web-id and names
* once able to automate the process, then can make a new dataframe with all info

### Bio functions to be used once inside a rower profile page

Rower firstname

In [12]:
first_name = BeautifulSoup(r.text, 'html').find("span", {"class": "sidearm-roster-player-first-name"})
rower_fn = str(first_name).split(">")[1].split('<')[0]
rower_fn

'Hannah'

Rower lastname

In [13]:
last_name = BeautifulSoup(r.text, 'html').find("span", {"class": "sidearm-roster-player-last-name"})
rower_ln = str(last_name).split(">")[1].split('<')[0]
rower_ln

'Valenty'

Rower bio information

In [14]:
bio = BeautifulSoup(r.text, 'html').find_all("span", {"class": False, "aria-live": False,
                                                      "data-bind": False})[0:3]

rower_bio = list(map(lambda b: str(b).split(">")[1].split('<')[0], bio))
rower_bio

['Senior', 'Carnegie, Pa.', 'Our Lady of the Sacred Heart']

Automate biographical data collection

In [15]:
def bio_collect(req):

    # rower first name
    first_name = BeautifulSoup(req.text, 'html').find("span", {"class": "sidearm-roster-player-first-name"})
    rower_fn = str(first_name).split(">")[1].split('<')[0]

    # rower last name
    last_name = BeautifulSoup(req.text, 'html').find("span", {"class": "sidearm-roster-player-last-name"})
    rower_ln = str(last_name).split(">")[1].split('<')[0]

    # rower grade, high school, hometown
    bio = BeautifulSoup(req.text, 'html').find_all("span", {"class": False, "aria-live": False,
                                                        "data-bind": False})[0:3]
    rower_bio = list(map(lambda b: str(b).split(">")[1].split('<')[0], bio))

    return rower_fn, rower_ln, rower_bio

In [30]:
# go through each season
bio_dict = {}
bio_list = []
for year in range(2021, 2025):
    #year_bio_df = pd.DataFrame(columns=['firstname','lastname', 'bio'])
    inner_bio_dict = {}
    # reformat three columns for requesting access
    for athlete in range(1, len(rower_id_df[f'season-{year}'])+1):
        insert = rower_id_df[f'season-{year}'].loc[athlete]
        # request access
        r = request_access(f'{insert[2]}-{insert[1]}/{insert[0]}')
        # get bio information
        f, l, b = bio_collect(req=r)
        # add to dataframe
        #year_bio_df.loc[athlete] = [f, l, b]
        # add to inner dictionary
        #bio_dict[f'{f}_{l}'] = {'first_name': f, 'last_name': l, 'bio': b, 'year': year}
        bio_list.append({'first_name': f, 'last_name': l, 'bio': b, 'year': year})
    # add to full dictionary
    #bio_dict.append(inner_bio_dict) #year_bio_df


C:\Users\Valenty\AppData\Local\Temp\ipykernel_9904\3033763907.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  r = request_access(f'{insert[2]}-{insert[1]}/{insert[0]}')


Export bio information to CSV

In [32]:
# column names
field_names = ['first_name','last_name','bio', 'year']

with open('bio_list.csv', 'w') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=field_names)
    writer.writeheader()
    writer.writerows(bio_list)